# 01 Setup and Data

The Jupyter version of the framework-py tour, condensed into four notebooks. Run them in
order: later ones use the data this one adds to the catalog. Each cell says what to look
for. Compare with the Quarto series in `../py-project`.

## Start with scaffold()

`scaffold()` finds the project root, loads `.env` and the settings, records the run in
`framework.db`, and puts `functions/` on the import path. It does not change the working
directory: Jupyter runs this notebook from `notebooks/`, and Framework still resolves
paths from the root.

In [ ]:
import os
import shutil
import warnings

import framework as fw

root = fw.scaffold()
print("cwd: ", os.getcwd())
print("root:", root)

`status()` prints an overview of the project. The git lines describe the outer
`framework-test-projects` repository, since this project has no `.git` of its own.

In [ ]:
fw.status()

## Settings

`settings()` reads a value by dot-notation key; a one-part key is looked up in
`directories` first. This project uses the Jupyter workflow, so `notebook_format` is
`jupyter` and `make_notebook()` creates `.ipynb` files.

In [ ]:
print(fw.settings("notebooks"))
print(fw.settings("scaffold.notebook_format"))
print(fw.settings("connections.default_database"))
print(fw.settings("missing.key", default="(missing)"))

`settings_read()` merges the split files under `settings/` into one dict.

In [ ]:
fw.settings_read()["connections"]["databases"]

## Copy the raw files in

The study CSVs live outside the project in `shared/csv/`. They are copied to
`inputs/raw/` so catalog paths stay relative to the project.

In [ ]:
shared_csv = root.parent.parent / "shared" / "csv"
raw = root / fw.settings("inputs_raw")
for name in ("participants", "visits", "labs", "survey"):
    shutil.copy(shared_csv / f"{name}.csv", raw / f"{name}.csv")
sorted(p.name for p in raw.iterdir())

## The data catalog

`data_add()` registers a file under a name derived from its folder and records its
SHA-256 hash. Files are locked unless `locked=False`.

In [ ]:
for name in ("participants", "visits", "labs"):
    fw.data_add(raw / f"{name}.csv")
fw.data_add(raw / "survey.csv", locked=False);

`data_list()` shows the catalog; `data_info()` shows how one name is read.

In [ ]:
fw.data_list()

In [ ]:
fw.data_info("inputs.raw.visits")

`data_read()` checks the hash and returns a pandas DataFrame. Note the missing blood
pressure readings.

In [ ]:
visits = fw.data_read("inputs.raw.visits")
participants = fw.data_read("inputs.raw.participants")
visits[["systolic_bp", "diastolic_bp"]].isna().sum()

## Clean and save

Join visits to participants and drop visits without a reading. `data_save()` picks the
folder from the name, writes Parquet (or CSV with `type="csv"`), and catalogs the file.

In [ ]:
bp = (
    visits.merge(participants, on="participant_id")
    .dropna(subset=["systolic_bp", "diastolic_bp"])
)
fw.data_save(bp, "inputs.intermediate.bp_visits")
fw.data_save(bp, "inputs.final.bp_analysis", type="csv")
fw.data_list()[["name", "path", "locked"]]

## Integrity checks

`data_verify()` compares a file with its recorded hash.

In [ ]:
fw.data_verify("inputs.final.bp_analysis")

A changed locked file stops the next read with `IntegrityError`. This uses a small demo
file so the real inputs stay intact, then accepts the change by saving again.

In [ ]:
fw.data_save(participants.head(5), "inputs.intermediate.integrity_demo", type="csv")
path = fw.data_info("inputs.intermediate.integrity_demo").path
with open(path, "a") as f:
    f.write("999,1,2024-01-01,50,male,25.0,0,control\n")

try:
    fw.data_read("inputs.intermediate.integrity_demo")
except fw.IntegrityError as error:
    print(error)

fw.data_save(participants.head(5), "inputs.intermediate.integrity_demo", type="csv");

An unlocked file warns with `DataChangedWarning` and reads the new rows.

In [ ]:
fw.data_save(participants.head(5), "inputs.intermediate.integrity_demo", type="csv", locked=False)
with open(path, "a") as f:
    f.write("998,2,2024-01-02,61,female,31.0,1,intervention\n")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    rows = len(fw.data_read("inputs.intermediate.integrity_demo"))
print(rows, [w.category.__name__ for w in caught])
fw.data_save(participants.head(5), "inputs.intermediate.integrity_demo", type="csv");